# CNN 1D Univariate

In this section we will implement a convolutional neural network for timeseries forecasting. 

The CNN 1D (1D Convolutional Neural Network) Forecaster is designed for multivariate time series forecasting. It uses convolutional layers to capture local temporal patterns and dependencies efficiently, processing sequences in parallel rather than sequentially.

## Architecture

```bash
Input (seq_length, input_size)
    ↓
Transpose (input_size, seq_length)
    ↓
Conv1D Block 1
  ├─ Conv1D (input_size → hidden_size, kernel=3)
  ├─ ReLU
  └─ MaxPool1D
    ↓
Conv1D Block 2
  ├─ Conv1D (hidden_size → hidden_size, kernel=3)
  ├─ ReLU
  └─ MaxPool1D
    ↓
Conv1D Block 3
  └─ (same structure)
    ↓
Adaptive Average Pooling (→ 1)
    ↓
Flatten
    ↓
Fully Connected (hidden_size → hidden_size)
    ↓
ReLU + Dropout
    ↓
Fully Connected (hidden_size → 1)
    ↓
Output (1 prediction)
```

## Layer Breakdown

- **Conv1D Blocks:** 3 stacked convolutional blocks (configurable)
- **Kernel Size:** 3 (captures patterns across 3 consecutive timesteps)
- **Hidden Size:** 64 filters per layer (default)
- **MaxPooling:** Applied after each convolution with stride=1, padding=1
- **Adaptive Average Pooling:** Reduces variable-length sequences to fixed size
- **Fully Connected Layers:** Two FC layers with ReLU activation between them
- **Dropout:** Applied before final output layer
- **Output Layer:** Single neuron producing 1-step forecast

## Advantages

- **Parallel Processing:** Can process entire sequence at once (faster than RNN/LSTM/GRU)
- **Fewer Parameters:** Generally requires fewer parameters than RNN-based models
- **No Vanishing Gradients:** No recurrent connections means no vanishing gradient problem

## Limitations

- **Limited Long-Range Dependencies:** Receptive field grows slowly with depth; may miss long-term patterns
- **Fixed Kernel Size:** Kernel size (3) determines the temporal context window
- **Less Interpretable:** Harder to understand what patterns each filter captures
- **Sequence Length Sensitivity:** Very short sequences may not benefit from multiple conv layers

## When to Use

- Need fast training and inference
- Local temporal patterns are important (e.g., sudden spikes, dips)
- Dataset is moderate to large (>5,000 samples)
- Sequences are short to medium length (5-50 timesteps)
- Parallel processing capability is valuable

## Key Hyperparameters

| Parameter | Default | Description |
|-----------|---------|-------------|
| hidden_size | 64 | Number of convolutional filters - more filters capture more patterns |
| num_layers | 3 | Number of convolutional blocks - depth increases receptive field |
| kernel_size | 3 | Size of convolution window - fixed at 3 timesteps |
| dropout | 0.2 | Dropout rate for regularization |

## Receptive Field

With 3 layers and kernel size 3:
- **Layer 1:** Sees 3 timesteps
- **Layer 2:** Sees 5 timesteps  
- **Layer 3:** Sees 7 timesteps

The receptive field grows by 2 timesteps per layer with kernel=3.


## Model

In [ ]:
import torch 
import torch.nn as nn

In [ ]:
class CNN1DForecaster(nn.Module):
    """
    1D CNN model for MULTIVARIATE time series forecasting.
    Architecture: 
        Conv1D blocks (Conv -> ReLU -> MaxPool) -> 
        Adaptive Average Pooling -> Flatten -> 
        Fully Connected -> Dropout -> Output
    
    CNNs can capture local patterns and temporal dependencies efficiently.
    Uses multiple kernel sizes to capture patterns at different scales.
    Processes sequences in parallel (unlike RNN/LSTM/GRU).
    """
    def __init__(self, input_size, hidden_size=64, num_layers=3, dropout=0.2):
        """
        Args:
            input_size: Number of input features (Value + year + month + one-hot)
            hidden_size: Number of filters in conv layers
            num_layers: Number of convolutional blocks (minimum 1)
            dropout: Dropout rate
        """
        super(CNN1DForecaster, self).__init__()
        
        self.input_size = input_size
        self.hidden_size = hidden_size
        self.num_layers = max(1, num_layers)
        
        # Convolutional layers
        conv_layers = []
        
        # First conv block: input_size -> hidden_size
        conv_layers.extend([
            nn.Conv1d(in_channels=input_size, out_channels=hidden_size, 
                     kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool1d(kernel_size=2, stride=1, padding=1)
        ])
        
        # Additional conv blocks: hidden_size -> hidden_size
        for i in range(1, self.num_layers):
            conv_layers.extend([
                nn.Conv1d(in_channels=hidden_size, out_channels=hidden_size, 
                         kernel_size=3, padding=1),
                nn.ReLU(),
                nn.MaxPool1d(kernel_size=2, stride=1, padding=1)
            ])
        
        self.conv_blocks = nn.Sequential(*conv_layers)
        
        # Adaptive pooling to fixed size output
        self.adaptive_pool = nn.AdaptiveAvgPool1d(1)
        
        # Fully connected layers
        self.fc1 = nn.Linear(hidden_size, hidden_size)
        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(dropout)
        self.fc2 = nn.Linear(hidden_size, 1)
    
    def forward(self, x):
        # x shape: (batch_size, seq_length, input_size)
        
        # Conv1d expects (batch_size, channels, seq_length)
        # Transpose from (batch, seq, features) to (batch, features, seq)
        x = x.transpose(1, 2)  # (batch_size, input_size, seq_length)
        
        # Apply convolutional blocks
        x = self.conv_blocks(x)  # (batch_size, hidden_size, seq_length')
        
        # Adaptive pooling to reduce to (batch_size, hidden_size, 1)
        x = self.adaptive_pool(x)  # (batch_size, hidden_size, 1)
        
        # Flatten
        x = x.squeeze(-1)  # (batch_size, hidden_size)
        
        # Fully connected layers
        x = self.fc1(x)  # (batch_size, hidden_size)
        x = self.relu(x)
        x = self.dropout(x)
        out = self.fc2(x)  # (batch_size, 1)
        
        return out


### Model Results without Exogenous Features

### Model Results with Exogenous Features